# 可变形注意力

> 目标：理解为什么它比标准注意力更高效、它是如何围绕参考点进行稀疏采样的，以及如何用 PyTorch 写一个简化版实现。

本文按下面的顺序展开：
1. 标准注意力的计算瓶颈
2. 可变形注意力的核心思想
3. 数学形式与张量维度
4. 基于 `grid_sample` 的简化代码实现
5. 一个最小可运行示例

可变形注意力最早在 Deformable DETR 中大规模使用，后续 BEVFormer 的空间交叉注意力本质上也是围绕参考点对多视角图像特征做稀疏采样与聚合。

## 1. 为什么标准注意力不够高效

> 对图像特征图做全局注意力时，每个 query 都要和所有 key 位置交互，代价非常高。

设特征图被摊平后长度为 $N$，query 个数为 $M$，标准注意力需要构造一个 $M \times N$ 的相关性矩阵，时间复杂度通常写作 $O(MN)$。

> 在视觉任务里，真正有用的信息往往只集中在少量局部区域。

可变形注意力的核心判断是：
- 没必要让每个 query 去看全部空间位置。
- 只要围绕一个“参考点”去采样少量关键位置，就能抓住主要信息。
- 这些采样位置不是固定卷积核，而是由网络动态预测，因此具有“可变形”能力。

因此它同时具备：
- 稀疏性：每个 query 只看少量采样点。
- 自适应性：采样点位置由 query 内容决定。
- 连续性：采样点可以是浮点坐标，通过双线性插值从特征图中取值。

## 2. 可变形注意力的核心机制

> 一个 query 不再和整张特征图逐点做注意力，而是先确定一个参考点，再在其周围采样 $K$ 个位置。

对每个 query，通常会做四件事：
1. 预测一个参考点 `reference_point`，表示当前 query 大概应该关注哪里。
2. 预测若干个偏移量 `sampling_offsets`，决定从参考点再往哪些方向采样。
3. 预测这些采样点各自的权重 `attention_weights`。
4. 在特征图上对这些浮点坐标做双线性插值，并对采样结果加权求和。

其直觉可以理解为：
- 参考点负责“粗定位”。
- 偏移量负责“细搜索”。
- 注意力权重负责“信息融合”。

如果使用多头注意力，那么每个 head 都会学习自己的一组偏移和权重，从而关注不同的局部结构。

在多尺度场景中，query 还会在多个 feature level 上分别采样，最后再把各层结果汇总。这就是常见的 multi-scale deformable attention。

## 3. 数学表达

> 下面先写单尺度版本，再说明如何扩展到多尺度。

设：
- query 表示为 $q_i$，共有 $M$ 个 query。
- value 特征图为 $x \in \mathbb{R}^{H \times W \times C}$。
- 第 $i$ 个 query 的参考点为 $p_i \in [0, 1]^2$，这里用归一化坐标表示。
- 第 $m$ 个 head、第 $k$ 个采样点的偏移量为 $\Delta p_{imk}$。
- 对应权重为 $A_{imk}$。

则单尺度可变形注意力可以写成：

$$
\mathrm{DeformAttn}(q_i, x) = \sum_{m=1}^{N_h} W_m \left( \sum_{k=1}^{K} A_{imk} \cdot x\big(p_i + \Delta p_{imk}\big) \right)
$$

其中：
- $N_h$ 是注意力头数。
- $K$ 是每个 head 的采样点个数。
- $x(p)$ 表示在连续坐标 $p$ 处对特征图做双线性插值后的结果。

如果是多尺度版本，设第 $l$ 层特征图为 $x^l$，则进一步写成：

$$
\mathrm{MSDeformAttn}(q_i) = \sum_{m=1}^{N_h} W_m \left( \sum_{l=1}^{L} \sum_{k=1}^{K} A_{imlk} \cdot x^l\big(p_i^l + \Delta p_{imlk}\big) \right)
$$

这里和标准注意力最本质的区别是：
- 标准注意力对所有 key 做显式匹配。
- 可变形注意力只访问少量可学习采样点。

因此当 $K \ll HW$ 时，计算量会明显下降。

## 4. 从张量角度理解实现流程

> 下面实现一个“单尺度 + 多头 + 每头多个采样点”的简化版本。

为了便于学习，我们约定输入形状如下：
- `query`: `(B, Nq, C)`
- `value`: `(B, H, W, C)`
- `reference_points`: `(B, Nq, 2)`，坐标范围是 `[0, 1]`

整个前向过程可以拆成 6 步：
1. 用线性层从 `query` 预测 `sampling_offsets`。
2. 用线性层从 `query` 预测 `attention_weights`。
3. 把参考点与偏移量相加，得到最终采样坐标。
4. 把采样坐标映射到 `grid_sample` 需要的 `[-1, 1]` 坐标系。
5. 用 `grid_sample` 在特征图上做双线性采样。
6. 按权重加权求和，再把多个 head 拼回输出通道。

注意两件容易混淆的事：
- `reference_points` 往往是归一化坐标，而不是像素坐标。
- `grid_sample` 的最后一维顺序是 `(x, y)`，对应 `(W, H)`。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SimplifiedDeformableAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, num_points):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError("embed_dim 必须能被 num_heads 整除")

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_points = num_points
        self.head_dim = embed_dim // num_heads

        self.value_proj = nn.Linear(embed_dim, embed_dim)
        self.offset_proj = nn.Linear(embed_dim, num_heads * num_points * 2)
        self.weight_proj = nn.Linear(embed_dim, num_heads * num_points)
        self.output_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, query, value, reference_points):
        """
        query: (B, Nq, C)
        value: (B, H, W, C)
        reference_points: (B, Nq, 2), 范围为 [0, 1]
        """
        batch_size, num_query, _ = query.shape
        _, height, width, _ = value.shape

        projected_value = self.value_proj(value)
        projected_value = projected_value.view(
            batch_size, height, width, self.num_heads, self.head_dim
        )
        projected_value = projected_value.permute(0, 3, 4, 1, 2).contiguous()
        projected_value = projected_value.view(
            batch_size * self.num_heads, self.head_dim, height, width
        )

        sampling_offsets = self.offset_proj(query)
        sampling_offsets = sampling_offsets.view(
            batch_size, num_query, self.num_heads, self.num_points, 2
        )

        attention_weights = self.weight_proj(query)
        attention_weights = attention_weights.view(
            batch_size, num_query, self.num_heads, self.num_points
        )
        attention_weights = F.softmax(attention_weights, dim=-1)

        normalizer = query.new_tensor([width, height]).view(1, 1, 1, 1, 2)
        sampling_locations = reference_points[:, :, None, None, :] + sampling_offsets / normalizer

        sampling_grid = sampling_locations * 2.0 - 1.0
        sampling_grid = sampling_grid.permute(0, 2, 1, 3, 4).contiguous()
        sampling_grid = sampling_grid.view(
            batch_size * self.num_heads, num_query, self.num_points, 2
        )

        sampled_value = F.grid_sample(
            projected_value,
            sampling_grid,
            mode="bilinear",
            padding_mode="zeros",
            align_corners=False,
        )
        sampled_value = sampled_value.view(
            batch_size, self.num_heads, self.head_dim, num_query, self.num_points
        )
        sampled_value = sampled_value.permute(0, 3, 1, 4, 2).contiguous()

        attention_weights = attention_weights.unsqueeze(-1)
        output = (sampled_value * attention_weights).sum(dim=3)
        output = output.view(batch_size, num_query, self.embed_dim)
        output = self.output_proj(output)
        return output

In [ ]:
batch_size = 2
num_query = 6
height, width = 20, 30
embed_dim = 64
num_heads = 8
num_points = 4

torch.manual_seed(0)

module = SimplifiedDeformableAttention(
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_points=num_points,
)

query = torch.randn(batch_size, num_query, embed_dim)
value = torch.randn(batch_size, height, width, embed_dim)
reference_points = torch.rand(batch_size, num_query, 2)

output = module(query, value, reference_points)
print("query shape:", query.shape)
print("value shape:", value.shape)
print("reference_points shape:", reference_points.shape)
print("output shape:", output.shape)

## 5. 代码实现逐段解释

> 这一版代码不是官方 CUDA 优化实现，但核心思想和真实工程版是一致的。

关键实现点如下：
- `value_proj`：先把输入特征投影到 attention 使用的通道空间。
- `offset_proj`：从 query 预测每个 head、每个采样点的二维偏移量。
- `weight_proj`：从 query 预测采样点权重，再用 `softmax` 归一化。
- `sampling_locations`：把参考点和偏移量相加，得到最终采样位置。
- `grid_sample`：在连续坐标上对特征图进行双线性插值。
- 最后按权重聚合，并通过 `output_proj` 输出。

如果你已经熟悉标准多头注意力，可以这样类比：
- 标准注意力中的 `QK^T` 用来在所有位置里寻找相关点。
- 可变形注意力直接跳过“全量搜索”，改成“直接预测该去哪里采样”。

这也是它速度更快、对高分辨率视觉特征更友好的原因。

## 6. 和 BEVFormer 中用法的关系

> 在 BEVFormer 里，可变形注意力不是孤立模块，而是服务于 BEV query 与多相机图像特征之间的信息交互。

可以这样理解：
- 每个 BEV query 对应鸟瞰图上的一个网格位置。
- 这个 BEV 位置会通过相机几何关系投影到不同视角图像上。
- 投影后的点可以看作参考点。
- 然后在参考点周围再学习少量偏移进行采样。
- 来自不同相机、不同尺度的采样结果再被融合回这个 BEV query。

所以 BEVFormer 中的空间交叉注意力，本质上是在做：
- 几何先验提供粗定位。
- 可变形采样提供细粒度对齐。
- 注意力权重负责多视角信息融合。

如果后面要继续补这份笔记，一个自然的下一步是：
1. 加入 multi-scale deformable attention 的张量版本实现。
2. 再进一步补上“BEV query 如何投影到各相机平面”的几何推导。